# 面试问题：推荐/广告在线探索怎样实现 LinUCB，并处理 propensity、延迟反馈和安全约束？

**一句话回答**：contextual bandit 每轮观察上下文、选一个动作并只看到该动作奖励。LinUCB 为每个 arm 维护 ridge regression 的 `A,b`，用预测均值加不确定性 bonus 探索；线上必须记录候选集、context/策略版本、action 和 propensity，延迟反馈按 event ID join，安全/库存约束在选动作前过滤，离线用 IPS/SNIPS 时检查 support 与有效样本量。

本 Notebook 用 NumPy 从零实现 LinUCB、随机 baseline、延迟回流、regret、IPS 和折扣适应漂移。

In [ ]:
from dataclasses import dataclass
from collections import deque, Counter
import hashlib, json, math
import numpy as np

SEED89=8901; rng89=np.random.default_rng(SEED89)
ARMS89=4; DIM89=5
theta89=rng89.normal(size=(ARMS89,DIM89)); theta89/=np.linalg.norm(theta89,axis=1,keepdims=True)
assert theta89.shape==(4,5) and np.allclose(np.linalg.norm(theta89,axis=1),1)
assert SEED89==8901
assert hashlib.sha256(theta89.tobytes()).hexdigest()

## 1. Decision log 合同

每次决策记录 decision ID、主体、context、完整可行动作集、chosen arm、每 arm score、chosen propensity、policy/feature version 和时间。奖励事件另带 reward time/definition。缺候选集或 propensity 的日志无法可靠做反事实评估。

context 必须是决策时可用的快照；把点击后特征或未来库存写进日志会产生泄漏。

In [ ]:
@dataclass(frozen=True)
class Decision89:
    decision_id:str; context:tuple; candidates:tuple; action:int; propensity:float; policy_version:str; event_time:int
    def __post_init__(self):
        if not self.decision_id or len(self.context)!=DIM89 or self.action not in self.candidates or not 0<self.propensity<=1 or not self.policy_version: raise ValueError("decision_contract")
d89=Decision89("d1",tuple(np.ones(DIM89)),tuple(range(ARMS89)),0,.25,"random-v1",0)
assert d89.action==0 and len(d89.candidates)==4
try: Decision89("",tuple(np.ones(DIM89)),(0,),1,0,"",0); raise AssertionError("invalid decision accepted")
except ValueError as e: assert str(e)=="decision_contract"
assert math.isclose(sum([.25]*4),1)

## 2. 受控环境与 regret oracle

真实线上只观察 chosen reward，无法直接知道 regret。教学环境定义每 arm 期望奖励 `sigmoid(theta_a·x)`，oracle 选择最高期望 arm；Bernoulli 点击只用于更新。比较策略时预先生成 context 和 uniform noise，避免不同随机轨迹造成不公平。

合成 regret 只验证实现，不代表真实推荐效果。

In [ ]:
def sigmoid89(z): return 1/(1+np.exp(-np.clip(z,-30,30)))
T89=1800; contexts89=rng89.normal(size=(T89,DIM89)); reward_uniform89=rng89.random(T89)
expected89=sigmoid89(contexts89@theta89.T)
oracle_actions89=np.argmax(expected89,axis=1); oracle_values89=np.max(expected89,axis=1)
assert expected89.shape==(T89,ARMS89) and np.all((expected89>0)&(expected89<1))
assert len(np.unique(oracle_actions89))==ARMS89
assert np.allclose(expected89[np.arange(T89),oracle_actions89],oracle_values89)

## 3. 从零实现 LinUCB

每 arm 维护 `A=λI+Σxxᵀ`、`b=Σrx`，参数 `θ=A⁻¹b`。选择分数 `θᵀx + α sqrt(xᵀA⁻¹x)`。初期 A 接近 λI，不确定性大；观察后对应方向 bonus 缩小。使用 `solve` 而不是显式存逆矩阵，更稳定。

tie-break 取最小 arm，保证重放确定性；候选过滤发生在 argmax 前。

In [ ]:
class LinUCB89:
    def __init__(self,arms,dim,alpha=1.,ridge=1.):
        if min(arms,dim,alpha,ridge)<=0: raise ValueError("linucb_contract")
        self.arms=arms; self.dim=dim; self.alpha=alpha; self.A=np.stack([np.eye(dim)*ridge for _ in range(arms)]); self.b=np.zeros((arms,dim)); self.counts=np.zeros(arms,int)
    def scores(self,x,candidates=None):
        x=np.asarray(x,float)
        if x.shape!=(self.dim,) or not np.isfinite(x).all(): raise ValueError("context_contract")
        candidates=range(self.arms) if candidates is None else candidates; out={}
        for a in candidates:
            theta=np.linalg.solve(self.A[a],self.b[a]); uncertainty=math.sqrt(max(0.,float(x@np.linalg.solve(self.A[a],x)))); out[a]=float(theta@x+self.alpha*uncertainty)
        return out
    def select(self,x,candidates=None):
        scores=self.scores(x,candidates); return max(scores,key=lambda a:(scores[a],-a)),scores
    def update(self,x,arm,reward,discount=1.):
        if arm<0 or arm>=self.arms or reward not in (0,1) or not 0<discount<=1: raise ValueError("update_contract")
        self.A[arm]*=discount; self.b[arm]*=discount; self.A[arm]+=np.outer(x,x); self.b[arm]+=reward*x; self.counts[arm]+=1
probe_policy89=LinUCB89(4,5); action_probe89,scores_probe89=probe_policy89.select(np.ones(5))
assert action_probe89==0 and len(scores_probe89)==4
probe_policy89.update(np.ones(5),0,1)
assert probe_policy89.counts[0]==1 and np.all(np.linalg.eigvalsh(probe_policy89.A[0])>0)

## 4. 与 uniform random 比较累计 regret

两个策略面对同一 context；random propensity=1/4，LinUCB 本例确定选择，在线反事实日志应加入显式随机化才能得到非零 propensity/support。更新只使用 chosen arm 的点击。

后半段 rolling regret 应低于 random，且所有 arm 被探索过。

In [ ]:
def run_policy89(kind):
    policy=LinUCB89(ARMS89,DIM89,alpha=.7); local_rng=np.random.default_rng(8910); regrets=[]; rewards=[]; actions=[]
    for t,x in enumerate(contexts89):
        a=int(local_rng.integers(ARMS89)) if kind=="random" else policy.select(x)[0]
        p=expected89[t,a]; r=int(reward_uniform89[t]<p); regrets.append(oracle_values89[t]-p); rewards.append(r); actions.append(a)
        if kind!="random": policy.update(x,a,r)
    return np.array(regrets),np.array(rewards),np.array(actions),policy
regret_random89,reward_random89,action_random89,_=run_policy89("random")
regret_ucb89,reward_ucb89,action_ucb89,policy89=run_policy89("ucb")
assert regret_ucb89[600:].mean()<regret_random89[600:].mean()*.75
assert reward_ucb89.mean()>reward_random89.mean() and all(policy89.counts>0)
assert len(regret_ucb89)==T89 and np.all(regret_ucb89>=-1e-12)

## 5. 延迟反馈与 pending decision store

点击可能几秒回流，购买可能数天回流。决策时保存不可变 context/action；reward 到达再 update。不能用 reward 时的最新 context。重复 reward event 以 event ID 去重，超出 attribution window 的事件按定义丢弃或用于长期指标。

下面按不同 delay 排队，验证乱序回流仍只更新一次。

In [ ]:
delay_policy89=LinUCB89(ARMS89,DIM89,.5); pending89=[]; seen_rewards89=set(); logged89={}
for t in range(120):
    x=contexts89[t]; a,_=delay_policy89.select(x); did=f"d{t}"; logged89[did]=(x.copy(),a); reward=int(reward_uniform89[t]<expected89[t,a]); pending89.append((t+(t%7),did,reward,f"event-{t}"))
    for due,old_did,old_reward,eid in sorted([e for e in pending89 if e[0]<=t]):
        if eid not in seen_rewards89:
            ox,oa=logged89[old_did]; delay_policy89.update(ox,oa,old_reward); seen_rewards89.add(eid)
    pending89=[e for e in pending89 if e[3] not in seen_rewards89]
for due,did,reward,eid in sorted(pending89):
    if eid not in seen_rewards89: x,a=logged89[did]; delay_policy89.update(x,a,reward); seen_rewards89.add(eid)
assert len(seen_rewards89)==120 and delay_policy89.counts.sum()==120
assert len(logged89)==120 and len({x for x in seen_rewards89})==120
assert all(np.isfinite(delay_policy89.A[a]).all() for a in range(ARMS89))

## 6. 用 epsilon wrapper 产生可记录 propensity

纯 deterministic UCB 对未选动作 propensity 为 0，不适合一般离线反事实评估。可在安全候选内使用 epsilon-greedy：以 `1-epsilon` 选 greedy，以 epsilon 均匀探索；greedy 动作概率为 `1-epsilon+epsilon/K`，其他为 `epsilon/K`。

采样、记录和离线重算必须使用同一候选集与 policy version；过滤后候选数变化会改变 propensity。

In [ ]:
def epsilon_probs89(scores,epsilon):
    if not 0<epsilon<1 or not scores: raise ValueError("epsilon_contract")
    arms=sorted(scores); greedy=max(arms,key=lambda a:(scores[a],-a)); probs={a:epsilon/len(arms) for a in arms}; probs[greedy]+=1-epsilon; return probs,greedy
probs89,greedy89=epsilon_probs89(policy89.scores(contexts89[10]),.1)
assert math.isclose(sum(probs89.values()),1.) and probs89[greedy89]>.9
assert all(p>0 for p in probs89.values())
safe_probs89,safe_greedy89=epsilon_probs89(policy89.scores(contexts89[10],(0,2)),.1)
assert set(safe_probs89)=={0,2} and safe_greedy89 in {0,2}

## 7. IPS/SNIPS 离线评估与 support

logging policy 随机选择 arm 并记录 propensity 1/4。评估 target deterministic policy 时，只保留 logging action 与 target action 相同的记录，权重 `1/p`; IPS 除以总日志数，SNIPS 除以权重和。若某 target action 在某 context 下 propensity 为 0，无法无偏评估。

报告 ESS，权重尾部过大时应增加探索或限制 target policy。

In [ ]:
eval_rng89=np.random.default_rng(8920); logging_actions89=eval_rng89.integers(0,ARMS89,T89); observed_rewards89=(reward_uniform89<expected89[np.arange(T89),logging_actions89]).astype(float); target_actions89=action_ucb89
match89=logging_actions89==target_actions89; weights89=match89.astype(float)*ARMS89
ips89=float(np.mean(weights89*observed_rewards89)); snips89=float(np.sum(weights89*observed_rewards89)/np.sum(weights89)); oracle_target89=float(np.mean(expected89[np.arange(T89),target_actions89])); ess89=float(weights89.sum()**2/np.sum(weights89**2))
assert abs(ips89-oracle_target89)<.08 and abs(snips89-oracle_target89)<.06
assert 300<ess89<600 and match89.any()
assert np.all((weights89==0)|(weights89==ARMS89))

## 8. 安全候选、非平稳与发布

库存、年龄、合规和频控先产生 allowed candidates，bandit 只能在其中选；探索不能越过硬约束。环境漂移时历史 A/b 可能拖慢适应，可用滑动窗口或折扣，但折扣也增加方差。离线 replay 和小流量 canary 验证 reward、regret proxy、覆盖和约束违规率。

policy artifact 保存 A/b、alpha/ridge/discount、feature order、arm catalog 和训练水位；加载时校验摘要。

In [ ]:
safe_candidates89=(0,2); safe_action89,safe_scores89=policy89.select(contexts89[0],safe_candidates89)
assert safe_action89 in safe_candidates89 and set(safe_scores89)==set(safe_candidates89)
state_sha89=hashlib.sha256(policy89.A.tobytes()+policy89.b.tobytes()).hexdigest(); manifest89={"schema":1,"policy":"linucb","arms":ARMS89,"dim":DIM89,"alpha":.7,"feature_order":[f"x{i}" for i in range(DIM89)],"state_sha256":state_sha89,"reward":"click_24h_v1"}
digest89=hashlib.sha256(json.dumps(manifest89,sort_keys=True,separators=(",",":")).encode()).hexdigest()
assert len(state_sha89)==len(digest89)==64
assert manifest89["arms"]==theta89.shape[0] and manifest89["dim"]==theta89.shape[1]
assert policy89.counts.sum()==T89
print({"random_regret":round(regret_random89.mean(),3),"ucb_regret":round(regret_ucb89.mean(),3),"ips":round(ips89,3),"ess":round(ess89,1)})

## 9. 面试收束、参考与练习

回答闭环：decision/reward 合同 → LinUCB 推导 → regret oracle → 延迟/重复反馈 → propensity/support/ESS → 安全候选 → 非平稳折扣 → artifact/灰度。不要把普通监督排序器加随机噪声就称为完整 bandit 系统。

练习：实现 Thompson Sampling；给 LinUCB 加 arm feature 的 hybrid 版本；模拟 reward delay 与 censoring；比较 discount 对突变后的恢复速度。

参考：[LinUCB 论文](https://arxiv.org/abs/1003.0146)、[Counterfactual Risk Minimization](https://arxiv.org/abs/1502.02362)、[Bandit Algorithms 教材](https://tor-lattimore.com/downloads/book/book.pdf)。